In [2]:
%pip install -U langchain langchain-core langchain-community langchain-openai langchain-openrouter langchain-qdrant langchain-text-splitters qdrant-client python-dotenv pypdf pymupdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

--------
## Loader

In [2]:
from langchain_core.documents import Document

doc = Document(
    page_content=(
        "Full-time employees receive 24 days of paid annual leave per calendar year. "
        "Up to 5 unused days may be carried over into the next year."
    ),
    metadata={"filename": "employee_handbook.pdf", "page": 1, "area": "policies"},
)

print(doc.page_content)
print(doc.metadata)

Full-time employees receive 24 days of paid annual leave per calendar year. Up to 5 unused days may be carried over into the next year.
{'filename': 'employee_handbook.pdf', 'page': 1, 'area': 'policies'}


In [3]:
from langchain_community.document_loaders import TextLoader

notes = TextLoader("knowledge/wiki/onboarding.md", encoding="utf-8").load()

print(len(notes))                  # 1 file, so 1 Document
print(notes[0].metadata)           # {'source': 'knowledge/wiki/onboarding.md'}
print(notes[0].page_content[:200])

C:\Users\shmahajan\AppData\Local\Temp\ipykernel_11888\91933484.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


1
{'source': 'knowledge/wiki/onboarding.md'}
# Engineering Onboarding

Welcome to Zomato Engineering. This page covers your first weeks. Ask in #new-joiners if anything is unclear.

## Day one

- Collect your laptop and access card from the IT d


In [4]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("knowledge/policies/employee_handbook.pdf").load()

print(f"Pages loaded: {len(pages)}")
print(pages[1].page_content[:240])
print(pages[1].metadata)

Pages loaded: 8
4. Working hours
Standard hours are 10:00 to 19:00, Monday to Friday, with a one-hour lunch break. Core collaboration hours are 11:00 to
17:00, when you are expected to be available for meetings. Teams that support live operations, such as 
{'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-02T18:18:26+00:00', 'title': 'handbook.html', 'moddate': '2026-10-02T18:18:26+00:00', 'source': 'knowledge/policies/employee_handbook.pdf', 'total_pages': 8, 'page': 1, 'page_label': '2'}


In [8]:
%pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import time
from langchain_community.document_loaders import PDFPlumberLoader, PyMuPDFLoader, PyPDFLoader

path = "knowledge/policies/expense_policy.pdf"      # a PDF that contains a table
candidates = {"pypdf": PyPDFLoader, "pymupdf": PyMuPDFLoader, "pdfplumber": PDFPlumberLoader}

for name, loader_cls in candidates.items():
    started = time.perf_counter()
    pages = loader_cls(path).load()
    seconds = time.perf_counter() - started
    chars = sum(len(p.page_content) for p in pages)
    print(f"{name:>10}: {len(pages)} pages, {chars:,} characters, {seconds:.2f}s")
    print("            sample:", pages[2].page_content[:90].replace("\n", " | "))

     pypdf: 5 pages, 6,239 characters, 0.07s
            sample: 6. Domestic travel | Item Rule | Flights Economy class. Book at least 7 days ahead where possi
   pymupdf: 5 pages, 6,239 characters, 0.01s
            sample: 6. Domestic travel | Item | Rule | Flights | Economy class. Book at least 7 days ahead where possi
pdfplumber: 5 pages, 6,244 characters, 0.20s
            sample: 6. Domestic travel | Item Rule | Flights Economy class. Book at least 7 days ahead where possi


In [13]:
%pip install unstructured pi_heif unstructured-inference pytesseract

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
%pip install -U "pillow<10.4.0" --force-reinstall

  Using cached pillow-10.3.0.tar.gz (46.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      Traceback (most recent call last):
        File "c:\Users\shmahajan\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "c:\Users\shmahajan\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Users\shmahajan\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "C:\Users\shmahajan\AppData\Local\Temp\pip-build-env-bs9v81a_\overla

In [22]:
from langchain_community.document_loaders import UnstructuredPDFLoader

scan = UnstructuredPDFLoader(
    "knowledge/scans/vendor_contract.pdf",
    mode="elements",            # titles, paragraphs and tables as separate Documents
    strategy="ocr_only",        # force OCR instead of reading a text layer
).load()

print(len(scan), "elements")
print(scan[0].metadata.get("category"), "|", scan[0].page_content[:80])

ImportError: cannot import name '_Ink' from 'PIL._typing' (c:\Users\shmahajan\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\_typing.py)

In [23]:
from langchain_community.document_loaders import CSVLoader

faq = CSVLoader(
    "knowledge/faq/support_faq.csv",
    encoding="utf-8",
    metadata_columns=["product", "category"],
).load()

print(len(faq), "rows")
print(faq[0].page_content)
print(faq[0].metadata)

63 rows
question: How do I get a refund for a cancelled order?
answer: Refunds for cancelled orders go back to the original payment method within 5 to 7 working days. Zomato credits are refunded instantly.
{'source': 'knowledge/faq/support_faq.csv', 'row': 0, 'product': 'Zomato App', 'category': 'Refunds'}


------

## Chunking

In [5]:
from langchain_community.document_loaders import PyMuPDFLoader

handbook_docs = PyMuPDFLoader("knowledge/policies/employee_handbook.pdf").load()
handbook_text = "\n\n".join(doc.page_content for doc in handbook_docs)

print(f"Handbook: {len(handbook_docs)} pages, {len(handbook_text):,} characters")

C:\Users\shmahajan\AppData\Local\Temp\ipykernel_21712\785489719.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Handbook: 8 pages, 11,016 characters


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,         # maximum characters per chunk
    chunk_overlap=120,      # characters shared between neighbouring chunks
    add_start_index=True,   # records where in the page each chunk began
)

chunks = splitter.split_documents(handbook_docs)

print(f"Split {len(handbook_docs)} pages into {len(chunks)} chunks")
print("\nFirst chunk:")
print(chunks[1].page_content)
print(len(chunks[1].page_content))
print("\nMetadata:", chunks[0].metadata)

Split 8 pages into 16 chunks

First chunk:
Simple and honest. Say what you mean, share bad news early and prefer plain language over jargon.
Respect for everyone. Every person who works with Zomato, from a rider to a director, deserves courtesy and
dignity.
3. Employment basics
New employees serve a probation period of 6 months. Confirmation depends on a probation review with your manager
in month 5. During probation either side may end employment with 15 days' notice. Employment types are Full-time,
Fixed-term and Intern. Benefits in this handbook apply to full-time employees unless stated otherwise. Interns receive a
stipend and access to the learning portal but not leave or insurance benefits.
662

Metadata: {'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-02T18:18:26+00:00', 'source': 'knowledge/policies/employee_handbook.pdf', 'file_path

In [7]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

text = open("knowledge/wiki/onboarding.md", encoding="utf-8").read()

# 1. cuts only at blank lines, then merges paragraphs up to the size limit
by_char = CharacterTextSplitter(separator="\n\n", chunk_size=500, chunk_overlap=50)

# 2. tries blank lines, then newlines, then spaces, so oversized paragraphs still get split
by_recursive = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# 3. sizes in tokens, matching what the embedding model actually counts
by_tokens = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=120, chunk_overlap=15
)

for name, splitter in [("character", by_char), ("recursive", by_recursive), ("tokens", by_tokens)]:
    pieces = splitter.split_text(text)
    longest = max(len(p) for p in pieces)
    print(f"{name:>10}: {len(pieces):>3} chunks, longest = {longest} characters")

 character:  15 chunks, longest = 479 characters
 recursive:  15 chunks, longest = 479 characters
    tokens:  12 chunks, longest = 596 characters


In [8]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
)

markdown = open("knowledge/wiki/onboarding.md", encoding="utf-8").read()
sections = header_splitter.split_text(markdown)

print(len(sections), "sections")
print(sections[2].metadata)
print(sections[2].page_content[:150])

size_cap = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120, add_start_index=True)
chunks = size_cap.split_documents(sections)      # sections that already fit stay as they are

print(f"\n{len(sections)} sections → {len(chunks)} chunks after size cap")

17 sections
{'h1': 'Engineering Onboarding', 'h2': 'Accounts and access'}
You will receive these accounts on day one: company email, Slack, GitHub, Jira, Confluence and the internal Zomato Admin dashboard. Production access 

17 sections → 17 chunks after size cap


In [10]:
import hashlib
import uuid


def chunk_id(chunk) -> str:
    digest = hashlib.sha1(chunk.page_content.encode("utf-8")).hexdigest()
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{digest}"))


for chunk in chunks:
    chunk.metadata["tenant_id"] = "zomato"

ids = [chunk_id(c) for c in chunks]
print(len(ids), len(set(ids)))       # equal, unless two chunks in one file are identical

17 17


In [11]:
import os
from langchain_openai import OpenAIEmbeddings


embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_base="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

texts = [
    "Full-time employees receive 24 days of paid annual leave.",
    "Client meals are reimbursed up to the per-person limit with a receipt.",
    "Restart the payments service from the runbook, never from the console.",
]

vectors = embeddings.embed_documents(texts)
print("Number of vectors:", len(vectors))       # 3
print("Vector dimension:",  len(vectors[0]))    # 1536 for text-embedding-3-small
print("First 5 values:",    vectors[0][:5])

query_vector = embeddings.embed_query("How many vacation days do I get?")
print("Query vector dimension:", len(query_vector))

Number of vectors: 3
Vector dimension: 1536
First 5 values: [-0.01296234130859375, 0.03570556640625, 0.0192108154296875, 0.029022216796875, -0.0161285400390625]
Query vector dimension: 1536


In [ ]:
from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=ids,                          # the stable IDs from the previous step
    location=":memory:",
    collection_name="helpdesk_kb",
)

print("Vector store ready")

In [12]:
# Connect to a running Qdrant server (uncomment after starting Docker)

# from langchain_qdrant import QdrantVectorStore
# vector_store = QdrantVectorStore.from_documents(
#     documents=chunks,
#     embedding=embeddings,
#     ids=ids,
#     url="http://localhost:6333",
#     collection_name="helpdesk_kb",
# )

# Reconnect to an existing collection (don't re-embed everything on restart):
# vector_store = QdrantVectorStore.from_existing_collection(
#     embedding=embeddings,
#     collection_name="helpdesk_kb",
#     url="http://localhost:6333",
# )

# Full control — create the collection yourself and hand the client to LangChain:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")
dimension = len(embeddings.embed_query("probe"))
if not client.collection_exists("helpdesk_kb"):
    client.create_collection(
        collection_name="helpdesk_kb",
        vectors_config=models.VectorParams(size=dimension, distance=models.Distance.COSINE),
    )
vector_store = QdrantVectorStore(client=client, collection_name="helpdesk_kb", embedding=embeddings)

print(vector_store.collection_name)

helpdesk_kb


In [13]:
import hashlib
import uuid

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_id(chunk) -> str:
    digest = hashlib.sha1(chunk.page_content.encode("utf-8")).hexdigest()
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{chunk.metadata['filename']}|{digest}"))


# Load
handbook_pages = PyMuPDFLoader("knowledge/policies/employee_handbook.pdf").load()
for doc in handbook_pages:
    doc.metadata["filename"] = "employee_handbook.pdf"
    doc.metadata["area"] = "policies"
    doc.metadata["doc_type"] = "pdf"
    doc.metadata["tenant_id"] = "zomato"
print(f"Total pages: {len(handbook_pages)}")
print(handbook_pages[0])

# Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120, add_start_index=True)
handbook_chunks = splitter.split_documents(handbook_pages)
handbook_ids = [chunk_id(c) for c in handbook_chunks]

print(f"Chunks: {len(handbook_chunks)}, unique IDs: {len(set(handbook_ids))}")

# Upsert — uses the embeddings and vector_store created above
vector_store.add_documents(handbook_chunks, ids=handbook_ids)
print(f"Points in collection: {vector_store.client.count('helpdesk_kb').count}")

Total pages: 8
page_content='Zomato Employee Handbook
Version 4.2 · Effective 1 April 2026 · Owner: People Operations
1. Welcome to Zomato
Zomato connects millions of customers with restaurants and delivery partners every day. We are a technology company at
heart, and our work touches food, logistics, payments and support. This handbook explains how we work, what you can
expect from us and what we expect from you. If anything here conflicts with your employment contract, the contract
applies.
2. Our values
Customer obsession. Start with what the customer, restaurant partner or delivery partner needs, then work
backwards.
Ownership. If you see a problem, you own it until it is fixed or handed over clearly.
Speed with care. We move fast, and we do not cut corners on food safety, payments or privacy.
Simple and honest. Say what you mean, share bad news early and prefer plain language over jargon.
Respect for everyone. Every person who works with Zomato, from a rider to a director, deserve

In [14]:
question = "How many vacation days do I get?"

results = vector_store.similarity_search(question, k=3)

for rank, doc in enumerate(results, start=1):
    print(f"\n--- Result {rank} ---")
    print(doc.metadata.get("filename"), "page:", doc.metadata.get("page"))
    print(doc.page_content[:250])


--- Result 1 ---
employee_handbook.pdf page: 2
Floating
holidays
2 days per year
For local or personal festivals. Apply 7 days ahead.
Study leave
Up to 10 days
For exams linked to an approved course, with manager
approval.
Sabbatical
Up to 3 months
unpaid
After 4 years of service, subject to busi

--- Result 2 ---
employee_handbook.pdf page: 1
Up to 5 unused days may be carried over into the next year. Anything above 5 days lapses on 31 March.
Leave of more than 3 consecutive days needs manager approval at least 14 days ahead. Shorter leave needs
approval at least 2 working days ahead, exc

--- Result 3 ---
employee_handbook.pdf page: 2
7. Parental leave
Type
Paid
leave
Notes
Maternity
26 weeks
Can start up to 8 weeks before the due date. A further 4 weeks of unpaid
leave can be requested.
Paternity
4 weeks
Take within 6 months of the birth. Can be split into two blocks.
Adoption
12


In [15]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

retrieved = retriever.invoke("What is the process for reimbursing a client dinner?")

print(f"Retrieved {len(retrieved)} documents")
for doc in retrieved:
    print(" ", doc.metadata.get("filename"), "page:", doc.page_content)

Retrieved 4 documents
  employee_handbook.pdf page: 11. Benefits
Health insurance: cover of ₹5 lakh for you, your spouse and up to two children, paid fully by the company.
Parents can be added at a discounted group premium paid by you.
Accident and life cover: accidental death cover of ₹50 lakh and term life cover of 3 times your annual fixed
pay.
Meal credits: ₹3,000 per month in Zomato credits, usable on the app. Unused credits expire at month end.
Learning budget: ₹40,000 per year for courses, books and conferences, with manager approval. Unspent
budget does not carry over.
Wellness: 6 free counselling sessions per year through the employee assistance programme, plus a monthly gym
or fitness reimbursement of up to ₹1,500.
Phone and internet: reimbursement of up to ₹1,000 per month for work-related use.
Relocation: for approved transfers, a relocation package covering travel, temporary housing for 30 days and
  employee_handbook.pdf page: An exit interview is held in your final week,

In [ ]:
from qdrant_client import models

only_policies = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.area",          # LangChain stores metadata under "metadata"
            match=models.MatchValue(value="policies"),
        )
    ]
)

results = vector_store.similarity_search(
    "Can I expense a client dinner?",
    k=4,
    filter=only_policies,
)

for doc in results:
    print(doc.metadata.get("filename"), "area:", doc.page_content)

-----
## Call LLM

In [16]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="deepseek/deepseek-v4.1-flash",   # any model id from openrouter.ai/models
    temperature=0,                 # 0 keeps answers focused and repeatable
)

response = llm.invoke([
    ("system", "You are a concise assistant."),
    ("human", "Explain RAG in one sentence."),
])

print(response.content)

RAG (Retrieval-Augmented Generation) is a technique where a language model retrieves relevant documents or data from an external knowledge source and feeds them into its prompt, so it can generate answers grounded in up-to-date or domain-specific information rather than relying solely on its training.


In [17]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Zomato's internal help-desk assistant.
Answer using only the context below.

Rules:
- If the answer is not in the context, say you don't know.
- Never invent policies, amounts or procedures.

Context:
{context}""",
    ),
    ("human", "{question}"),
])

print("Prompt template created with placeholders: {context} and {question}")

Prompt template created with placeholders: {context} and {question}


In [18]:
def format_docs(docs) -> str:
    return "\n\n---\n\n".join(
        f"[{i}] {d.metadata.get('filename', 'unknown')}\n{d.page_content}"
        for i, d in enumerate(docs, start=1)
    )

print("format_docs defined")

format_docs defined


In [20]:
question = "What is the policy on work from home?"
print(f"Question: {question}")

# Step 1: retrieve
retrieved_docs = retriever.invoke(question)
print(f"Step 1 — Retrieved {len(retrieved_docs)} docs")

# Step 2: format
context = format_docs(retrieved_docs)
print(f"Step 2 — Context: {len(context)} characters")

# Step 3: build the prompt
messages = prompt.invoke({"context": context, "question": question})
print(f"Step 3 — Prompt messages: {len(messages.to_messages())}")

# Step 4: generate
response = llm.invoke(messages)
print(f"\nStep 4 — Answer:\n{response.content}")

Question: What is the policy on work from home?
Step 1 — Retrieved 4 docs
Step 2 — Context: 3084 characters
Step 3 — Prompt messages: 2

Step 4 — Answer:
**Hybrid work (Section 9 of the handbook)**

- Most corporate teams follow a hybrid model: **3 days in the office (Tuesday to Thursday)** and **2 days remote**. Managers may adjust this for team needs and will tell you in writing.
- **Customer Support and Delivery Operations** roles are office or shift based and are **not covered** by the hybrid model.
- **Working from another city** for more than 10 working days in a quarter needs **written approval from HR**.
- **Working from another country is not permitted** without Legal and Finance approval, because of tax and visa rules.
- Remote days still follow **core hours**, and you should be **reachable on Slack**.
- A **one-time work-from-home setup allowance of ₹15,000** is available for a desk and chair, claimable through the **Expense Portal**.


------
## LCEL

In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context":  retriever | format_docs,  # question → Documents → context string
        "question": RunnablePassthrough(),    # question → question, unchanged
    }
    | prompt
    | llm
    | StrOutputParser()                       # AIMessage → plain string
)

print(rag_chain.invoke("How many days of work from how can I have?"))

Based on the handbook, under the hybrid model most corporate teams get **2 days remote** per week (with 3 days in the office, Tuesday to Thursday).

A few caveats from the context:
- Managers may adjust this for team needs, and will tell you in writing.
- Customer Support and Delivery Operations roles are office or shift based and are **not** covered by the hybrid model.
- Working from another city for more than 10 working days in a quarter needs written approval from HR, and working from another country is not permitted without Legal and Finance approval.

I don't have any other information on remote/work-from-home day entitlements beyond this.


In [27]:
# Stream tokens as they arrive
print("Streaming answer:")
for token in rag_chain.stream("error ZM-401 Policy"):
    print(token, end="", flush=True)
print()

Streaming answer:
I don't know. The context provided doesn't mention an error code "ZM-401" or any related policy, so I can't tell you what it means or how to resolve it.

If you can share more details about where you saw this error (for example, which system or tool), I can point you to the relevant part of the handbook if it's covered there. Otherwise, you may want to raise it with IT or People Operations.
